# PII Guardrail Model Training

Fine-tune Llama 3.2-1B for PII detection using Unsloth (QLoRA 4-bit).

**Output Format:**
```json
{
  "flagged": true/false,
  "confidence": 1-10,
  "entities": [{"type": "...", "value": "...", "start": N, "end": N}],
  "reason": "..."
}
```

**Risk Levels:** HIGH (9-10), MEDIUM (6-8), LOW (1-5)

**Supported PII Types:** PERSON, EMAIL, PHONE, IN_AADHAAR, IN_PAN, US_SSN, CREDIT_CARD


In [ ]:
# Cell 1: Install Dependencies
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes


In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Cell 3: Configuration
import os

# Paths - Update these to match your Google Drive structure
DATA_DIR = "/content/drive/MyDrive/pii-guardrail-data"
OUTPUT_DIR = "/content/drive/MyDrive/pii-guardrail-model"

TRAIN_FILE = os.path.join(DATA_DIR, "train_v2.jsonl")
EVAL_FILE = os.path.join(DATA_DIR, "eval_v2.jsonl")

# Model config
MODEL_NAME = "unsloth/Llama-3.2-1B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
LORA_R = 16
LORA_ALPHA = 16

# Training config
EPOCHS = 3
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Train file: {TRAIN_FILE}")
print(f"Eval file: {EVAL_FILE}")


In [ ]:
# Cell 4: Enhanced System Prompt (Risk-Aware)
SYSTEM_PROMPT = """You are a PII (Personally Identifiable Information) detection and risk assessment system.

TASK: Analyze text, identify PII entities, and assess their risk level.

=== STRICT EXTRACTION RULES ===
1. ONLY report PII that EXACTLY exists in the input text
2. The "value" field must be a VERBATIM substring from the input
3. "start" and "end" must be correct character positions
4. If NO PII exists, return flagged: false with empty entities
5. NEVER invent, guess, or hallucinate PII values

=== RISK-BASED CLASSIFICATION ===

HIGH RISK (flag immediately, confidence 9-10):
- Email addresses (any format)
- Phone numbers (with country code or local)
- Government IDs: Aadhaar, PAN, SSN, Passport
- Credit/Debit card numbers
- Full residential addresses
- Bank account numbers
- Full name + contact detail combination

MEDIUM RISK (flag with caution, confidence 6-8):
- Full name + city/location
- Full name + workplace/company
- Employee/Customer ID numbers
- Date of birth with other identifiers
- IP addresses in user context

LOW RISK (usually don't flag, confidence 1-5):
- First name only (no surname)
- Role-based mentions (CEO, Manager, Candidate)
- General city/country without identity link
- Public figures in news context
- Fictional characters
- Generic reference numbers (Order ID, Ticket #)

=== PII TYPES TO DETECT ===
- PERSON: Full names (first + last name together)
- EMAIL: Email addresses (user@domain.com)
- PHONE: Phone numbers (+91 XXXXX XXXXX, XXX-XXX-XXXX)
- IN_AADHAAR: Indian Aadhaar (12 digits: XXXX XXXX XXXX)
- IN_PAN: Indian PAN (ABCDE1234F format)
- US_SSN: US Social Security (XXX-XX-XXXX)
- CREDIT_CARD: Card numbers (13-19 digits)

=== OUTPUT FORMAT (JSON only) ===
{
  "flagged": true/false,
  "confidence": 1-10,
  "entities": [{"type": "TYPE", "value": "exact_text", "start": N, "end": N}],
  "reason": "brief explanation with risk assessment"
}

=== CONFIDENCE SCORING ===
- 9-10: HIGH RISK - Clear PII with explicit context ("My Aadhaar is", "Email:")
- 6-8: MEDIUM RISK - PII present but context is ambiguous
- 3-5: LOW RISK - Pattern matches but likely not sensitive
- 1-2: Very uncertain - Could be false positive

CRITICAL: If the value does not exist verbatim in the input, do NOT report it."""

print("Enhanced system prompt loaded.")
print(f"Length: {len(SYSTEM_PROMPT)} characters")


In [ ]:
# Cell 5: Load Training Data
import json

def load_jsonl(filepath):
    """Load JSONL file."""
    data = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

# Load data
print(f"Loading training data from: {TRAIN_FILE}")
train_data = load_jsonl(TRAIN_FILE)
print(f"Loading evaluation data from: {EVAL_FILE}")
eval_data = load_jsonl(EVAL_FILE)

print(f"\nLoaded {len(train_data)} training samples")
print(f"Loaded {len(eval_data)} evaluation samples")

# Analyze distribution
def analyze_distribution(data, name):
    flagged_true = 0
    flagged_false = 0
    pii_types = {}
    
    for item in data:
        assistant_msg = item['conversations'][2]['content']
        try:
            result = json.loads(assistant_msg)
            if result.get('flagged', False):
                flagged_true += 1
                for entity in result.get('entities', []):
                    pii_type = entity.get('type', 'UNKNOWN')
                    pii_types[pii_type] = pii_types.get(pii_type, 0) + 1
            else:
                flagged_false += 1
        except:
            pass
    
    print(f"\n{name} Distribution:")
    print(f"  flagged=true: {flagged_true} ({100*flagged_true/len(data):.1f}%)")
    print(f"  flagged=false: {flagged_false} ({100*flagged_false/len(data):.1f}%)")
    print(f"  PII Types: {pii_types}")

analyze_distribution(train_data, "Training")
analyze_distribution(eval_data, "Evaluation")


In [ ]:
# Cell 6: Format Dataset for Training
from datasets import Dataset

def format_conversation(item):
    """Format a single conversation for training."""
    conversations = item['conversations']
    
    # Use our enhanced system prompt instead of the one in the data
    formatted = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": conversations[1]['content']},
        {"role": "assistant", "content": conversations[2]['content']}
    ]
    
    return {"conversations": formatted}

# Format datasets
train_formatted = [format_conversation(item) for item in train_data]
eval_formatted = [format_conversation(item) for item in eval_data]

# Create HuggingFace datasets
train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(eval_formatted)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Eval dataset: {len(eval_dataset)} samples")

# Preview
print("\n--- Sample Training Example ---")
sample = train_dataset[0]['conversations']
print(f"User: {sample[1]['content'][:100]}...")
print(f"Assistant: {sample[2]['content'][:200]}...")


In [ ]:
# Cell 7: Load Model with Unsloth
from unsloth import FastLanguageModel
import torch

print(f"Loading model: {MODEL_NAME}")
print(f"Max sequence length: {MAX_SEQ_LENGTH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

print("\nApplying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print("Model loaded successfully!")
print(f"Trainable parameters: {model.print_trainable_parameters()}")


In [ ]:
# Cell 8: Training Configuration
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=50,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="none",
)

print("Training configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Gradient accumulation: {GRAD_ACCUM}")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Output: {OUTPUT_DIR}")


In [ ]:
# Cell 9: Train the Model
from unsloth.chat_templates import get_chat_template

# Apply chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

# Apply formatting
train_dataset_formatted = train_dataset.map(formatting_prompts_func, batched=True)
eval_dataset_formatted = eval_dataset.map(formatting_prompts_func, batched=True)

# Create trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("Starting training...")
print("="*60)

# Train
trainer_stats = trainer.train()

print("="*60)
print("Training complete!")
print(f"Total training time: {trainer_stats.metrics['train_runtime']:.2f}s")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")


In [ ]:
# Cell 10: Evaluation Functions
import re

def extract_json(text):
    """Extract JSON from model output."""
    try:
        return json.loads(text)
    except:
        pass
    
    patterns = [
        r'```json\s*([\s\S]*?)```',
        r'```\s*([\s\S]*?)```',
        r'(\{[\s\S]*\})'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            try:
                return json.loads(match.group(1))
            except:
                continue
    return None

def validate_entities(input_text, entities):
    """Validate that entities actually exist in input text (grounding check)."""
    valid_entities = []
    for entity in entities:
        value = entity.get('value', '')
        if value and value in input_text:
            actual_start = input_text.find(value)
            if actual_start >= 0:
                entity['start'] = actual_start
                entity['end'] = actual_start + len(value)
                valid_entities.append(entity)
    return valid_entities

def run_inference(text, validate=True):
    """Run inference on a single text."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Analyze for PII: "{text}"'}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    output_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    result = extract_json(output_text)
    
    # Apply grounding validation
    if validate and result and result.get('entities'):
        result['entities'] = validate_entities(text, result['entities'])
        if not result['entities']:
            result['flagged'] = False
            result['reason'] = "No valid PII after grounding check"
    
    return result, output_text

print("Evaluation functions loaded.")


In [ ]:
# Cell 11: Evaluate Model on Test Set
FastLanguageModel.for_inference(model)

print("Evaluating on eval dataset...")
print("="*60)

true_positives = 0
false_positives = 0
true_negatives = 0
false_negatives = 0
parse_errors = 0

eval_sample = eval_data[:200]

for i, item in enumerate(eval_sample):
    if i % 50 == 0:
        print(f"Progress: {i}/{len(eval_sample)}")
    
    user_msg = item['conversations'][1]['content']
    text_match = re.search(r'Analyze for PII: "(.*)"', user_msg)
    if not text_match:
        continue
    input_text = text_match.group(1)
    
    expected = json.loads(item['conversations'][2]['content'])
    expected_flagged = expected.get('flagged', False)
    
    result, _ = run_inference(input_text, validate=True)
    
    if result is None:
        parse_errors += 1
        continue
    
    predicted_flagged = result.get('flagged', False)
    
    if expected_flagged and predicted_flagged:
        true_positives += 1
    elif expected_flagged and not predicted_flagged:
        false_negatives += 1
    elif not expected_flagged and predicted_flagged:
        false_positives += 1
    else:
        true_negatives += 1

total = true_positives + true_negatives + false_positives + false_negatives
accuracy = (true_positives + true_negatives) / total if total > 0 else 0
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("\n" + "="*60)
print("EVALUATION RESULTS")
print("="*60)
print(f"Samples evaluated: {total}")
print(f"Parse errors: {parse_errors}")
print(f"\nConfusion Matrix:")
print(f"  True Positives:  {true_positives}")
print(f"  True Negatives:  {true_negatives}")
print(f"  False Positives: {false_positives}")
print(f"  False Negatives: {false_negatives}")
print(f"\nMetrics:")
print(f"  Accuracy:  {accuracy:.2%}")
print(f"  Precision: {precision:.2%}")
print(f"  Recall:    {recall:.2%}")
print(f"  F1 Score:  {f1:.2%}")


In [ ]:
#!/usr/bin/env python3
"""
PII Guardrail Interactive Test Script

Usage:
    In Google Colab after training:
    1. Copy this script into a new cell after Cell 10
    2. Run: results = run_all_tests(model, tokenizer, SYSTEM_PROMPT)
    3. Or interactive: interactive_mode(model, tokenizer, SYSTEM_PROMPT)

To add custom test cases, edit the TEST_CASES list below.
"""

import json
import re

# ============================================================================
# TEST CASES - Edit these to test different scenarios
# ============================================================================

TEST_CASES = [
    # Format: ("input text", expected_flagged, "description")
    # expected_flagged: True = should flag, False = should NOT flag, None = uncertain
    
    # === HIGH RISK (Should flag, confidence 9-10) ===
    ("My Aadhaar number is 1234 5678 9012", True, "Aadhaar with context"),
    ("Email me at john.doe@company.com", True, "Email address"),
    ("SSN: 123-45-6789", True, "US SSN"),
    ("Call me at +91 98765 43210", True, "Phone number"),
    ("PAN: ABCDE1234F", True, "Indian PAN"),
    ("Credit card: 4532 1234 5678 9012", True, "Credit card"),
    ("My Aadhaar is 9876 5432 1098", True, "Aadhaar explicit"),
    ("Contact priya.sharma@gmail.com for details", True, "Gmail address"),
    
    # === MEDIUM RISK (Should flag, confidence 6-8) ===
    ("Rahul Kumar works at TCS in Bangalore", True, "Name + company + location"),
    ("Contact Priya Sharma for the project", True, "Full name in context"),
    ("Employee ID: EMP12345 for Amit Verma", True, "Employee ID with name"),
    
    # === LOW RISK / NEGATIVES (Should NOT flag) ===
    ("The meeting is scheduled for tomorrow at 10 AM", False, "Generic meeting"),
    ("Order ID: ORD-2024-789456 shipped", False, "Order reference"),
    ("Error code 503 indicates server unavailable", False, "Error code"),
    ("The CEO announced new policies", False, "Role mention"),
    ("Harry Potter is a fictional character", False, "Fictional name"),
    ("Bangalore has heavy traffic today", False, "City only"),
    ("Reference number REF123456789", False, "Reference ID"),
    ("The capital of France is Paris", False, "Geography fact"),
    ("Product SKU: PROD-ABC-12345", False, "Product SKU"),
    ("Ticket #TKT-2024-001 has been resolved", False, "Ticket number"),
    ("The weather forecast predicts rain", False, "Weather"),
    ("Conference Room B is booked for 3 PM", False, "Room booking"),
    
    # === EDGE CASES (None = just report, don't judge) ===
    ("My number is 1234567890", None, "Ambiguous 10-digit number"),
    ("Contact support at help@", False, "Incomplete email"),
    ("John mentioned the deadline", None, "First name only"),
    ("ABCDE1234", False, "Partial PAN-like string"),
    ("The code is 1234-5678-9012", None, "Formatted number"),
    ("IP: 192.168.1.1", None, "IP address"),
    ("DOB: 15/08/1990", None, "Date of birth"),
    
    # === ADVERSARIAL CASES ===
    ("My A a d h a a r is 1 2 3 4 5 6 7 8 9 0 1 2", None, "Spaced out Aadhaar"),
    ("email: john dot doe at company dot com", None, "Obfuscated email"),
    ("Call nine eight seven six five four three two one zero", None, "Phone in words"),
    
    # === ADD YOUR CUSTOM TESTS HERE ===
    # ("your test input", True/False/None, "description"),
]

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def extract_json(text):
    """Extract JSON from model output."""
    try:
        return json.loads(text)
    except:
        pass
    
    patterns = [
        r'```json\s*([\s\S]*?)```',
        r'```\s*([\s\S]*?)```',
        r'(\{[\s\S]*\})'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            try:
                return json.loads(match.group(1))
            except:
                continue
    return None


def validate_entities(input_text, entities):
    """Validate that entities actually exist in input text (grounding check)."""
    valid_entities = []
    for entity in entities:
        value = entity.get('value', '')
        if value and value in input_text:
            actual_start = input_text.find(value)
            if actual_start >= 0:
                entity['start'] = actual_start
                entity['end'] = actual_start + len(value)
                valid_entities.append(entity)
    return valid_entities


def get_risk_level(confidence):
    """Map confidence to risk level."""
    if confidence >= 9:
        return "HIGH"
    elif confidence >= 6:
        return "MEDIUM"
    return "LOW"


def run_test(text, model, tokenizer, system_prompt, validate=True):
    """Run inference on a single text.
    
    Args:
        text: Input text to analyze
        model: The loaded model
        tokenizer: The tokenizer
        system_prompt: The system prompt
        validate: Whether to apply grounding validation
        
    Returns:
        tuple: (result_dict, raw_output_text)
    """
    import torch
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f'Analyze for PII: "{text}"'}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    output_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    result = extract_json(output_text)
    
    # Apply grounding validation
    if validate and result and result.get('entities'):
        original_count = len(result['entities'])
        result['entities'] = validate_entities(text, result['entities'])
        result['hallucinations_caught'] = original_count - len(result['entities'])
        if not result['entities']:
            result['flagged'] = False
            result['reason'] = "No valid PII after grounding check"
    
    # Add risk level
    if result:
        result['risk_level'] = get_risk_level(result.get('confidence', 5))
    
    return result, output_text


# ============================================================================
# MAIN TEST RUNNER
# ============================================================================

def run_all_tests(model, tokenizer, system_prompt, test_cases=None):
    """Run all test cases and print results.
    
    Args:
        model: The loaded model
        tokenizer: The tokenizer
        system_prompt: The system prompt
        test_cases: Optional list of test cases (uses TEST_CASES if None)
        
    Returns:
        dict: Results summary with passed/failed/uncertain counts
    """
    
    if test_cases is None:
        test_cases = TEST_CASES
    
    print("\n" + "="*80)
    print("PII GUARDRAIL TEST SUITE")
    print("="*80)
    print(f"Running {len(test_cases)} test cases...\n")
    
    results = {
        'passed': 0,
        'failed': 0,
        'uncertain': 0,
        'errors': 0,
        'details': []
    }
    
    for i, (text, expected, description) in enumerate(test_cases, 1):
        result, raw_output = run_test(text, model, tokenizer, system_prompt)
        
        if result is None:
            status = "ERROR"
            results['errors'] += 1
        elif expected is None:
            # Uncertain case - just report
            status = "CHECK"
            results['uncertain'] += 1
        elif result.get('flagged', False) == expected:
            status = "PASS"
            results['passed'] += 1
        else:
            status = "FAIL"
            results['failed'] += 1
        
        # Determine status emoji
        emoji = {"PASS": "[OK]", "FAIL": "[FAIL]", "CHECK": "[?]", "ERROR": "[ERR]"}.get(status, "?")
        
        print(f"{emoji} Test {i}: {description}")
        print(f"    Input: \"{text[:60]}{'...' if len(text) > 60 else ''}\"")
        
        expected_str = 'flagged' if expected else 'not flagged' if expected is False else 'uncertain'
        print(f"    Expected: {expected_str}")
        
        if result:
            flagged = result.get('flagged')
            confidence = result.get('confidence')
            risk = result.get('risk_level')
            print(f"    Got: flagged={flagged}, confidence={confidence}, risk={risk}")
            
            if result.get('entities'):
                for e in result['entities']:
                    print(f"        - {e.get('type')}: \"{e.get('value')}\"")
            
            if result.get('hallucinations_caught', 0) > 0:
                print(f"    [!] Hallucinations caught: {result['hallucinations_caught']}")
        else:
            print(f"    Got: Failed to parse output")
        
        print("-"*80)
        
        results['details'].append({
            'text': text,
            'expected': expected,
            'description': description,
            'result': result,
            'status': status
        })
    
    # Summary
    total = len(test_cases)
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"Total tests: {total}")
    print(f"[OK]   Passed:    {results['passed']} ({100*results['passed']/total:.0f}%)")
    print(f"[FAIL] Failed:    {results['failed']} ({100*results['failed']/total:.0f}%)")
    print(f"[?]    Uncertain: {results['uncertain']} ({100*results['uncertain']/total:.0f}%)")
    if results['errors'] > 0:
        print(f"[ERR]  Errors:    {results['errors']} ({100*results['errors']/total:.0f}%)")
    print("="*80)
    
    # Performance assessment
    if results['failed'] == 0 and results['errors'] == 0:
        print("\n[OK] All tests passed! Model is performing well.")
    elif results['failed'] <= 2:
        print("\n[OK] Minor issues. Review failed cases above.")
    else:
        print("\n[!] Multiple failures. Model may need more training or data.")
    
    return results


def interactive_mode(model, tokenizer, system_prompt):
    """Interactive testing mode - enter custom inputs.
    
    Args:
        model: The loaded model
        tokenizer: The tokenizer
        system_prompt: The system prompt
    """
    
    print("\n" + "="*80)
    print("INTERACTIVE MODE")
    print("="*80)
    print("Enter text to test (or 'quit' to exit):")
    print("Commands: 'quit', 'help', 'clear'\n")
    
    while True:
        try:
            text = input(">>> ").strip()
            
            if text.lower() in ['quit', 'exit', 'q']:
                break
            elif text.lower() == 'help':
                print("\nCommands:")
                print("  quit  - Exit interactive mode")
                print("  help  - Show this help")
                print("  clear - Clear screen (if supported)")
                print("\nJust type any text to test it for PII.\n")
                continue
            elif text.lower() == 'clear':
                print("\n" * 50)
                continue
            elif not text:
                continue
            
            result, raw_output = run_test(text, model, tokenizer, system_prompt)
            
            print("\nResult:")
            if result:
                print(json.dumps(result, indent=2))
            else:
                print("Failed to parse output")
                print(f"Raw output: {raw_output[:500]}")
            print()
            
        except KeyboardInterrupt:
            print("\n")
            break
        except Exception as e:
            print(f"Error: {e}")
    
    print("Exiting interactive mode.")


def quick_test(text, model, tokenizer, system_prompt):
    """Quick single test with formatted output.
    
    Args:
        text: Input text to analyze
        model: The loaded model
        tokenizer: The tokenizer
        system_prompt: The system prompt
        
    Returns:
        dict: The result
    """
    result, _ = run_test(text, model, tokenizer, system_prompt)
    
    print(f"\nInput: \"{text}\"")
    print("-" * 60)
    
    if result:
        print(f"Flagged: {result.get('flagged')}")
        print(f"Confidence: {result.get('confidence')}")
        print(f"Risk Level: {result.get('risk_level')}")
        print(f"Reason: {result.get('reason')}")
        
        if result.get('entities'):
            print("\nEntities:")
            for e in result['entities']:
                print(f"  - {e.get('type')}: \"{e.get('value')}\" [{e.get('start')}:{e.get('end')}]")
    else:
        print("Failed to parse model output")
    
    return result


# ============================================================================
# USAGE EXAMPLES (for Colab)
# ============================================================================

USAGE = """
=============================================================================
USAGE IN GOOGLE COLAB
=============================================================================

After running the training cells (1-10), add a new cell with:

# Option 1: Run all predefined tests
results = run_all_tests(model, tokenizer, SYSTEM_PROMPT)

# Option 2: Interactive mode (type your own inputs)
interactive_mode(model, tokenizer, SYSTEM_PROMPT)

# Option 3: Quick single test
quick_test("My email is test@example.com", model, tokenizer, SYSTEM_PROMPT)

# Option 4: Custom test cases
my_tests = [
    ("Custom input 1", True, "Should flag"),
    ("Custom input 2", False, "Should not flag"),
]
results = run_all_tests(model, tokenizer, SYSTEM_PROMPT, test_cases=my_tests)

=============================================================================
"""

# If running as standalone (not in Colab)
if __name__ == "__main__":
    print(USAGE)
    print("\nThis script is designed to run in Google Colab after training.")
    print("Copy the functions into a Colab cell and use as shown above.")



In [ ]:
# Cell 12: Test Inference - Negative Samples (Should NOT flag)
FastLanguageModel.for_inference(model)

negative_tests = [
    "The meeting yesterday focused on project milestones and delivery risks.",
    "Bangalore has seen increased traffic congestion over the past few years.",
    "The capital of France is Paris, which is known for its history.",
    "The error code 503 indicates that the service is unavailable.",
    "Order ID: ORD-2024-789456 has been shipped via express delivery.",
    "Reference number REF123456789 for your support ticket.",
    "The CEO announced new policies for remote work.",
    "Harry Potter is a fictional character from the books.",
    "Meeting scheduled for 10:30 AM in Conference Room B.",
    "The weather forecast predicts rain tomorrow afternoon."
]

print("="*70)
print("NEGATIVE SAMPLE TESTS (Should return flagged: false)")
print("="*70)

false_positives_count = 0
for text in negative_tests:
    result, _ = run_inference(text, validate=True)
    
    status = "PASS" if result and not result.get('flagged', True) else "FAIL"
    if "FAIL" in status:
        false_positives_count += 1
    
    print(f"\n[{status}] Input: {text[:50]}...")
    if result:
        print(f"  Flagged: {result.get('flagged')}, Confidence: {result.get('confidence')}")
    print("-"*70)

print(f"\nFalse Positive Rate: {false_positives_count}/{len(negative_tests)}")


In [ ]:
# Cell 13: Test Inference - Positive Samples (Should flag)
FastLanguageModel.for_inference(model)

positive_tests = [
    "Please contact John Smith at john.smith@email.com for details.",
    "My Aadhaar number is 2345 6789 0123 for verification.",
    "PAN card: ABCDE1234F belongs to the applicant.",
    "Call me at +91 98765 43210 if you need anything.",
    "SSN: 123-45-6789 is required for the application.",
    "Credit card 4532 1234 5678 9012 was charged.",
    "Send documents to priya.sharma@company.org please.",
    "Aadhaar: 9876 5432 1098 linked to the account.",
    "Contact Rahul Verma at 9876543210 for support.",
    "PAN BXYPK5678R is registered under this name."
]

print("="*70)
print("POSITIVE SAMPLE TESTS (Should return flagged: true)")
print("="*70)

false_negatives_count = 0
for text in positive_tests:
    result, _ = run_inference(text, validate=True)
    
    status = "PASS" if result and result.get('flagged', False) else "FAIL"
    if "FAIL" in status:
        false_negatives_count += 1
    
    print(f"\n[{status}] Input: {text}")
    if result:
        print(f"  Flagged: {result.get('flagged')}, Confidence: {result.get('confidence')}")
        for e in result.get('entities', []):
            print(f"  - {e.get('type')}: '{e.get('value')}'")
    print("-"*70)

print(f"\nFalse Negative Rate: {false_negatives_count}/{len(positive_tests)}")


In [ ]:
# Cell 14: Test Risk-Based Classification
FastLanguageModel.for_inference(model)

risk_tests = [
    # HIGH RISK - Should flag with confidence 9-10
    ("My Aadhaar is 1234 5678 9012", "HIGH"),
    ("Email me at john@company.com", "HIGH"),
    ("SSN: 123-45-6789", "HIGH"),
    
    # MEDIUM RISK - Should flag with confidence 6-8
    ("Rahul Kumar works at TCS in Bangalore", "MEDIUM"),
    ("Employee ID: EMP12345 for Priya", "MEDIUM"),
    
    # LOW RISK - Should NOT flag or low confidence
    ("John mentioned the project deadline", "LOW"),
    ("The CEO announced new policies", "LOW"),
    ("Harry Potter is a fictional character", "LOW"),
]

print("="*70)
print("RISK-BASED CLASSIFICATION TESTS")
print("="*70)

for text, expected_risk in risk_tests:
    result, _ = run_inference(text, validate=True)
    
    confidence = result.get('confidence', 0) if result else 0
    flagged = result.get('flagged', False) if result else False
    
    # Determine actual risk based on confidence
    if confidence >= 9:
        actual_risk = "HIGH"
    elif confidence >= 6:
        actual_risk = "MEDIUM"
    else:
        actual_risk = "LOW"
    
    status = "PASS" if (expected_risk == "LOW" and not flagged) or \
                       (expected_risk != "LOW" and flagged) else "CHECK"
    
    print(f"\n[{status}] Expected: {expected_risk}")
    print(f"Input: {text}")
    print(f"Flagged: {flagged}, Confidence: {confidence}, Risk: {actual_risk}")
    print("-"*70)


In [ ]:
# Cell 15: Production-Ready Inference Class
class PIIGuardrail:
    """Production-ready PII detection with risk-aware classification."""
    
    PATTERNS = {
        'EMAIL': r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        'PHONE': r'(?:\+?\d{1,3}[-.\s]?)?(?:\(?\d{2,4}\)?[-.\s]?)?\d{3,4}[-.\s]?\d{3,4}',
        'IN_AADHAAR': r'\d{4}[\s-]?\d{4}[\s-]?\d{4}',
        'IN_PAN': r'[A-Z]{5}\d{4}[A-Z]',
        'US_SSN': r'\d{3}-\d{2}-\d{4}',
        'CREDIT_CARD': r'\d{4}[\s-]?\d{4}[\s-]?\d{4}[\s-]?\d{4}',
    }
    
    def __init__(self, model, tokenizer, system_prompt):
        self.model = model
        self.tokenizer = tokenizer
        self.system_prompt = system_prompt
    
    def _quick_scan(self, text):
        for pattern in self.PATTERNS.values():
            if re.search(pattern, text):
                return True
        return False
    
    def _validate_entities(self, text, entities):
        valid = []
        for entity in entities:
            value = entity.get('value', '')
            if value and value in text:
                start = text.find(value)
                entity['start'] = start
                entity['end'] = start + len(value)
                valid.append(entity)
        return valid
    
    def _get_risk_level(self, confidence):
        if confidence >= 9:
            return "HIGH"
        elif confidence >= 6:
            return "MEDIUM"
        return "LOW"
    
    def analyze(self, text, use_prefilter=True):
        if use_prefilter and not self._quick_scan(text):
            return {
                "flagged": False,
                "confidence": 10,
                "entities": [],
                "reason": "No PII patterns detected (pre-filter)",
                "risk_level": "LOW"
            }
        
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f'Analyze for PII: "{text}"'}
        ]
        
        inputs = self.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
        ).to(self.model.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                inputs, max_new_tokens=512, do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        output_text = self.tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
        result = extract_json(output_text)
        
        if not result:
            return {
                "flagged": False, "confidence": 5, "entities": [],
                "reason": "Failed to parse model output", "risk_level": "LOW"
            }
        
        if result.get('entities'):
            result['entities'] = self._validate_entities(text, result['entities'])
            if not result['entities']:
                result['flagged'] = False
                result['reason'] = "No valid PII after grounding validation"
        
        result['risk_level'] = self._get_risk_level(result.get('confidence', 5))
        return result

# Create guardrail instance
guardrail = PIIGuardrail(model, tokenizer, SYSTEM_PROMPT)

print("PIIGuardrail class created.")
print("\nTest:")
result = guardrail.analyze("Contact john@example.com for help.")
print(json.dumps(result, indent=2))


In [ ]:
# Cell 16: Save Model for Deployment
print("Saving model for deployment...")
print("="*60)

# Save LoRA adapters
lora_path = os.path.join(OUTPUT_DIR, "lora_adapters")
model.save_pretrained(lora_path)
tokenizer.save_pretrained(lora_path)
print(f"LoRA adapters saved to: {lora_path}")

# Save merged model (16-bit)
merged_path = os.path.join(OUTPUT_DIR, "merged_16bit")
model.save_pretrained_merged(merged_path, tokenizer, save_method="merged_16bit")
print(f"Merged model (16-bit) saved to: {merged_path}")

# Save GGUF for llama.cpp (optional)
try:
    gguf_path = os.path.join(OUTPUT_DIR, "model.gguf")
    model.save_pretrained_gguf(gguf_path, tokenizer, quantization_method="q4_k_m")
    print(f"GGUF model saved to: {gguf_path}")
except Exception as e:
    print(f"GGUF export skipped: {e}")

print("\n" + "="*60)
print("MODEL EXPORT COMPLETE")
print("="*60)
print(f"\nFiles saved to: {OUTPUT_DIR}")
print("\nFor FastAPI deployment, use the 'merged_16bit' folder.")
print("For llama.cpp, use the 'model.gguf' file.")


In [ ]:
# Cell 18: COMPREHENSIVE TEST SUITE - All Edge Cases with Metrics
# ============================================================================
# This cell tests the model on various edge cases and provides detailed metrics
#
# IMPORTANT: You MUST run these cells FIRST (in order):
#   - Cell 1: Install dependencies (!pip install unsloth)
#   - Cell 7: Load model (this defines 'model' and 'tokenizer')
#   - Cell 4: System prompt (this defines 'SYSTEM_PROMPT')
#
# If your Colab session restarted, run Cell 1, then Cell 7, then this cell.
# ============================================================================

import json
import re
import time
import torch
from collections import defaultdict

# Check if model is loaded
try:
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
    print("[OK] Model is ready for inference")
except NameError:
    print("[ERROR] 'model' is not defined!")
    print("Please run Cell 1 (install) and Cell 7 (load model) first.")
    raise RuntimeError("Run Cell 1 and Cell 7 first!")
except Exception as e:
    print(f"[ERROR] {e}")
    print("Please run Cell 1 (install) first.")
    raise

# ============================================================================
# COMPREHENSIVE TEST DATASET
# ============================================================================

ALL_TEST_CASES = {
    # =========================================================================
    # HIGH RISK - SHOULD FLAG (confidence 9-10)
    # =========================================================================
    "HIGH_RISK_SHOULD_FLAG": [
        # Aadhaar
        ("My Aadhaar number is 1234 5678 9012", True, "Aadhaar with explicit context"),
        ("Aadhaar: 9876 5432 1098", True, "Aadhaar with label"),
        ("Please verify my aadhaar 2345-6789-0123", True, "Aadhaar with dashes"),
        ("UID: 1111 2222 3333 for KYC", True, "Aadhaar as UID"),
        
        # PAN
        ("PAN: ABCDE1234F", True, "PAN with label"),
        ("My PAN card number is BXYPK5678R", True, "PAN with context"),
        ("PAN CDEFG9876H for tax filing", True, "PAN in sentence"),
        
        # Email
        ("Email me at john.doe@company.com", True, "Standard email"),
        ("Contact priya.sharma@gmail.com for details", True, "Gmail address"),
        ("Send to support@example.org please", True, "Org email"),
        ("My email is test_user123@domain.co.in", True, "Indian domain email"),
        
        # Phone
        ("Call me at +91 98765 43210", True, "Indian mobile with country code"),
        ("Phone: 9876543210", True, "10-digit mobile"),
        ("Reach me at +1-555-123-4567", True, "US phone format"),
        ("Contact number: 022-12345678", True, "Landline with STD code"),
        
        # SSN
        ("SSN: 123-45-6789", True, "US SSN with label"),
        ("Social Security Number is 987-65-4321", True, "Full SSN context"),
        
        # Credit Card
        ("Credit card: 4532 1234 5678 9012", True, "Credit card with spaces"),
        ("Card number 4111111111111111", True, "Visa card continuous"),
        ("Pay with 5500-0000-0000-0004", True, "Mastercard with dashes"),
    ],
    
    # =========================================================================
    # MEDIUM RISK - SHOULD FLAG (confidence 6-8)
    # =========================================================================
    "MEDIUM_RISK_SHOULD_FLAG": [
        ("Rahul Kumar works at TCS in Bangalore", True, "Full name + company + city"),
        ("Contact Priya Sharma for the project", True, "Full name in context"),
        ("Employee ID: EMP12345 for Amit Verma", True, "Employee ID with name"),
        ("Sanjay Gupta from Mumbai office", True, "Name + city"),
        ("Dr. Anita Desai practices at Apollo Hospital", True, "Name + workplace"),
        ("Ravi Patel, Manager at Infosys", True, "Name + role + company"),
    ],
    
    # =========================================================================
    # LOW RISK / NEGATIVES - SHOULD NOT FLAG
    # =========================================================================
    "NEGATIVES_SHOULD_NOT_FLAG": [
        # Generic business text
        ("The meeting is scheduled for tomorrow at 10 AM", False, "Meeting schedule"),
        ("Please review the quarterly report by Friday", False, "Business request"),
        ("The project deadline has been extended", False, "Project update"),
        ("Our team will present the findings next week", False, "Team announcement"),
        
        # Reference numbers and IDs
        ("Order ID: ORD-2024-789456 shipped", False, "Order ID"),
        ("Reference number REF123456789", False, "Reference number"),
        ("Ticket #TKT-2024-001 has been resolved", False, "Ticket number"),
        ("Invoice INV-2024-0001 is due", False, "Invoice number"),
        ("Product SKU: PROD-ABC-12345", False, "Product SKU"),
        ("Tracking number: 1Z999AA10123456784", False, "Shipping tracking"),
        
        # Technical/Error codes
        ("Error code 503 indicates server unavailable", False, "Error code"),
        ("HTTP status 404 means not found", False, "HTTP status"),
        ("Version 2.3.1 is now available", False, "Version number"),
        ("Build #12345 passed all tests", False, "Build number"),
        
        # Role/Title mentions
        ("The CEO announced new policies", False, "CEO role"),
        ("Our CTO will lead the initiative", False, "CTO role"),
        ("The manager approved the request", False, "Manager role"),
        ("Contact the support team for help", False, "Team mention"),
        
        # Fictional/Public
        ("Harry Potter is a fictional character", False, "Fictional character"),
        ("Sherlock Holmes solved the mystery", False, "Fictional character"),
        ("Elon Musk announced new features", False, "Public figure"),
        ("Bill Gates founded Microsoft", False, "Public figure"),
        
        # Geography only
        ("Bangalore has heavy traffic today", False, "City mention"),
        ("The capital of France is Paris", False, "Geography fact"),
        ("Mumbai is the financial capital of India", False, "City description"),
        ("Weather in Delhi is hot today", False, "City + weather"),
        
        # Generic sentences
        ("The weather forecast predicts rain tomorrow", False, "Weather"),
        ("Conference Room B is booked for 3 PM", False, "Room booking"),
        ("The document has been uploaded successfully", False, "System message"),
        ("Please complete the form by end of day", False, "Instructions"),
    ],
    
    # =========================================================================
    # EDGE CASES - AMBIGUOUS (None = just report, model decides)
    # =========================================================================
    "EDGE_CASES_AMBIGUOUS": [
        ("My number is 1234567890", None, "Ambiguous 10-digit number"),
        ("Code: 1234-5678-9012", None, "Formatted number"),
        ("ID: 123456789012", None, "12-digit number"),
        ("Contact support at help@", False, "Incomplete email"),
        ("John mentioned the deadline", None, "First name only"),
        ("ABCDE1234", False, "Partial PAN-like string"),
        ("IP: 192.168.1.1", None, "IP address"),
        ("DOB: 15/08/1990", None, "Date of birth"),
        ("Account #123456", None, "Account number"),
        ("PIN: 1234", False, "4-digit PIN"),
        ("The code is ABC123", False, "Short alphanumeric"),
    ],
    
    # =========================================================================
    # ADVERSARIAL CASES - Obfuscated PII
    # =========================================================================
    "ADVERSARIAL_OBFUSCATED": [
        ("My A a d h a a r is 1 2 3 4 5 6 7 8 9 0 1 2", None, "Spaced Aadhaar"),
        ("email: john dot doe at company dot com", None, "Obfuscated email"),
        ("Call nine eight seven six five four three two one zero", None, "Phone in words"),
        ("P.A.N: A-B-C-D-E-1-2-3-4-F", None, "Spaced PAN"),
        ("aadhar num: 1234.5678.9012", None, "Aadhaar with dots"),
        ("ph: +91-9876-543-210", None, "Phone with extra dashes"),
    ],
    
    # =========================================================================
    # DANGEROUS NEGATIVES - Look like PII but are NOT
    # =========================================================================
    "DANGEROUS_NEGATIVES": [
        ("Transaction ID: 1234 5678 9012", False, "Transaction ID looks like Aadhaar"),
        ("Policy number POL1234567890", False, "Policy number"),
        ("Flight PNR: ABC123", False, "PNR code"),
        ("ISBN: 978-3-16-148410-0", False, "ISBN number"),
        ("MAC address: 00:1A:2B:3C:4D:5E", False, "MAC address"),
        ("IMEI: 123456789012345", False, "IMEI number"),
        ("Hexcode: #1234567890AB", False, "Hex color/code"),
        ("Serial: SN-1234-5678-9012", False, "Serial number"),
        ("UUID: 550e8400-e29b-41d4-a716-446655440000", False, "UUID"),
        ("Batch code: BATCH1234F", False, "Batch code looks like PAN"),
    ],
    
    # =========================================================================
    # MIXED CONTENT - Multiple elements
    # =========================================================================
    "MIXED_CONTENT": [
        ("Order ORD-123 for John Smith at john@email.com", True, "Order + Name + Email"),
        ("Meeting with Priya at 3 PM in Room 5", None, "Name + time + room"),
        ("Contact support@company.com or call 1800-123-456", True, "Email + toll-free"),
        ("User ID: USR123, Email: test@test.com", True, "User ID + Email"),
        ("Reference REF-456 for customer Rahul Kumar", True, "Reference + Name"),
    ],
}

# ============================================================================
# TEST RUNNER
# ============================================================================

def run_comprehensive_test(text, validate=True):
    """Run inference with validation."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f'Analyze for PII: "{text}"'}
    ]
    
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            inputs, max_new_tokens=512, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    inference_time = (time.time() - start_time) * 1000  # ms
    
    output_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    
    # Parse JSON
    result = None
    try:
        result = json.loads(output_text)
    except:
        for pattern in [r'```json\s*([\s\S]*?)```', r'```\s*([\s\S]*?)```', r'(\{[\s\S]*\})']:
            match = re.search(pattern, output_text)
            if match:
                try:
                    result = json.loads(match.group(1))
                    break
                except:
                    continue
    
    # Grounding validation
    hallucinations = 0
    if validate and result and result.get('entities'):
        original_count = len(result['entities'])
        valid_entities = []
        for entity in result['entities']:
            value = entity.get('value', '')
            if value and value in text:
                start = text.find(value)
                entity['start'] = start
                entity['end'] = start + len(value)
                valid_entities.append(entity)
        result['entities'] = valid_entities
        hallucinations = original_count - len(valid_entities)
        if not result['entities']:
            result['flagged'] = False
    
    return result, inference_time, hallucinations

# ============================================================================
# RUN ALL TESTS
# ============================================================================

print("="*80)
print("COMPREHENSIVE PII GUARDRAIL TEST SUITE")
print("="*80)
print(f"Running tests across {len(ALL_TEST_CASES)} categories...\n")

# Metrics storage
metrics = {
    'total': 0,
    'passed': 0,
    'failed': 0,
    'uncertain': 0,
    'errors': 0,
    'true_positives': 0,
    'true_negatives': 0,
    'false_positives': 0,
    'false_negatives': 0,
    'hallucinations_total': 0,
    'inference_times': [],
    'by_category': defaultdict(lambda: {'passed': 0, 'failed': 0, 'total': 0}),
    'failed_cases': [],
}

for category, test_cases in ALL_TEST_CASES.items():
    print(f"\n{'='*80}")
    print(f"CATEGORY: {category}")
    print(f"{'='*80}")
    
    for text, expected, description in test_cases:
        metrics['total'] += 1
        metrics['by_category'][category]['total'] += 1
        
        result, inf_time, hallucinations = run_comprehensive_test(text)
        metrics['inference_times'].append(inf_time)
        metrics['hallucinations_total'] += hallucinations
        
        if result is None:
            status = "ERROR"
            metrics['errors'] += 1
        elif expected is None:
            status = "CHECK"
            metrics['uncertain'] += 1
            metrics['by_category'][category]['passed'] += 1  # Count as passed for uncertain
        else:
            predicted = result.get('flagged', False)
            if predicted == expected:
                status = "PASS"
                metrics['passed'] += 1
                metrics['by_category'][category]['passed'] += 1
                if expected:
                    metrics['true_positives'] += 1
                else:
                    metrics['true_negatives'] += 1
            else:
                status = "FAIL"
                metrics['failed'] += 1
                metrics['by_category'][category]['failed'] += 1
                metrics['failed_cases'].append({
                    'text': text,
                    'expected': expected,
                    'got': predicted,
                    'description': description,
                    'category': category
                })
                if expected:
                    metrics['false_negatives'] += 1
                else:
                    metrics['false_positives'] += 1
        
        # Print result
        emoji = {"PASS": "[OK]", "FAIL": "[FAIL]", "CHECK": "[?]", "ERROR": "[ERR]"}.get(status, "?")
        print(f"\n{emoji} {description}")
        print(f"    Input: \"{text[:60]}{'...' if len(text) > 60 else ''}\"")
        
        if result:
            flagged = result.get('flagged')
            confidence = result.get('confidence')
            risk = "HIGH" if confidence and confidence >= 9 else "MEDIUM" if confidence and confidence >= 6 else "LOW"
            print(f"    Result: flagged={flagged}, confidence={confidence}, risk={risk}")
            if result.get('entities'):
                for e in result['entities'][:3]:  # Show max 3 entities
                    print(f"        - {e.get('type')}: \"{e.get('value')}\"")
            if hallucinations > 0:
                print(f"    [!] Hallucinations caught: {hallucinations}")
        else:
            print(f"    Result: PARSE ERROR")
        
        print(f"    Time: {inf_time:.1f}ms")

# ============================================================================
# FINAL METRICS SUMMARY
# ============================================================================

print("\n" + "="*80)
print("FINAL METRICS SUMMARY")
print("="*80)

# Calculate metrics
total_definite = metrics['true_positives'] + metrics['true_negatives'] + metrics['false_positives'] + metrics['false_negatives']
accuracy = (metrics['true_positives'] + metrics['true_negatives']) / total_definite if total_definite > 0 else 0
precision = metrics['true_positives'] / (metrics['true_positives'] + metrics['false_positives']) if (metrics['true_positives'] + metrics['false_positives']) > 0 else 0
recall = metrics['true_positives'] / (metrics['true_positives'] + metrics['false_negatives']) if (metrics['true_positives'] + metrics['false_negatives']) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
avg_inference_time = sum(metrics['inference_times']) / len(metrics['inference_times']) if metrics['inference_times'] else 0

print(f"""
╔══════════════════════════════════════════════════════════════════╗
║                    TEST RESULTS SUMMARY                          ║
╠══════════════════════════════════════════════════════════════════╣
║  Total Tests:        {metrics['total']:>4}                                       ║
║  Passed:             {metrics['passed']:>4} ({100*metrics['passed']/metrics['total']:.1f}%)                                ║
║  Failed:             {metrics['failed']:>4} ({100*metrics['failed']/metrics['total']:.1f}%)                                ║
║  Uncertain:          {metrics['uncertain']:>4} ({100*metrics['uncertain']/metrics['total']:.1f}%)                                ║
║  Parse Errors:       {metrics['errors']:>4}                                       ║
╠══════════════════════════════════════════════════════════════════╣
║                    CONFUSION MATRIX                              ║
╠══════════════════════════════════════════════════════════════════╣
║  True Positives:     {metrics['true_positives']:>4}  (Correctly flagged PII)              ║
║  True Negatives:     {metrics['true_negatives']:>4}  (Correctly passed clean text)        ║
║  False Positives:    {metrics['false_positives']:>4}  (Incorrectly flagged clean text)     ║
║  False Negatives:    {metrics['false_negatives']:>4}  (Missed actual PII)                  ║
╠══════════════════════════════════════════════════════════════════╣
║                    PERFORMANCE METRICS                           ║
╠══════════════════════════════════════════════════════════════════╣
║  Accuracy:           {accuracy:.2%}                                       ║
║  Precision:          {precision:.2%}                                       ║
║  Recall:             {recall:.2%}                                       ║
║  F1 Score:           {f1:.2%}                                       ║
╠══════════════════════════════════════════════════════════════════╣
║                    INFERENCE STATS                               ║
╠══════════════════════════════════════════════════════════════════╣
║  Avg Inference Time: {avg_inference_time:.1f}ms                                    ║
║  Min Inference Time: {min(metrics['inference_times']):.1f}ms                                    ║
║  Max Inference Time: {max(metrics['inference_times']):.1f}ms                                    ║
║  Hallucinations:     {metrics['hallucinations_total']:>4} (caught by grounding)             ║
╚══════════════════════════════════════════════════════════════════╝
""")

# Category breakdown
print("\nCATEGORY BREAKDOWN:")
print("-"*60)
for category, cat_metrics in metrics['by_category'].items():
    cat_pass_rate = 100 * cat_metrics['passed'] / cat_metrics['total'] if cat_metrics['total'] > 0 else 0
    status = "[OK]" if cat_pass_rate >= 80 else "[!]" if cat_pass_rate >= 60 else "[FAIL]"
    print(f"{status} {category}: {cat_metrics['passed']}/{cat_metrics['total']} ({cat_pass_rate:.0f}%)")

# Failed cases detail
if metrics['failed_cases']:
    print(f"\n{'='*80}")
    print(f"FAILED CASES DETAIL ({len(metrics['failed_cases'])} failures)")
    print("="*80)
    for i, case in enumerate(metrics['failed_cases'][:10], 1):  # Show max 10
        print(f"\n{i}. [{case['category']}] {case['description']}")
        print(f"   Input: \"{case['text'][:70]}...\"" if len(case['text']) > 70 else f"   Input: \"{case['text']}\"")
        print(f"   Expected: flagged={case['expected']}, Got: flagged={case['got']}")
    if len(metrics['failed_cases']) > 10:
        print(f"\n... and {len(metrics['failed_cases']) - 10} more failures")

# Final assessment
print("\n" + "="*80)
print("FINAL ASSESSMENT")
print("="*80)
if f1 >= 0.90:
    print("[OK] EXCELLENT: Model achieves 90%+ F1 score. Ready for production.")
elif f1 >= 0.80:
    print("[OK] GOOD: Model achieves 80%+ F1 score. Minor improvements may help.")
elif f1 >= 0.70:
    print("[!] ACCEPTABLE: Model achieves 70%+ F1 score. Consider more training data.")
else:
    print("[FAIL] NEEDS IMPROVEMENT: F1 score below 70%. Review training data and prompt.")

if metrics['false_positives'] > metrics['false_negatives']:
    print("[!] Model tends to OVER-FLAG (high false positives). Consider more negative samples.")
elif metrics['false_negatives'] > metrics['false_positives']:
    print("[!] Model tends to UNDER-FLAG (high false negatives). Consider more positive samples.")
else:
    print("[OK] False positives and false negatives are balanced.")

print("="*80)


In [ ]:
# Cell 17: Summary
print("""
╔══════════════════════════════════════════════════════════════════╗
║                    TRAINING COMPLETE                             ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  Model: Llama 3.2-1B fine-tuned for PII detection               ║
║                                                                  ║
║  Output Format:                                                  ║
║  {                                                               ║
║    "flagged": true/false,                                       ║
║    "confidence": 1-10,                                          ║
║    "entities": [{type, value, start, end}],                     ║
║    "reason": "...",                                             ║
║    "risk_level": "HIGH/MEDIUM/LOW"                              ║
║  }                                                               ║
║                                                                  ║
║  Risk Levels:                                                    ║
║  - HIGH (9-10): Email, Phone, Aadhaar, SSN, Credit Card         ║
║  - MEDIUM (6-8): Name+Location, Employee IDs                    ║
║  - LOW (1-5): First names only, public figures                  ║
║                                                                  ║
║  Supported PII Types:                                            ║
║  - PERSON, EMAIL, PHONE                                         ║
║  - IN_AADHAAR, IN_PAN                                           ║
║  - US_SSN, CREDIT_CARD                                          ║
║                                                                  ║
╠══════════════════════════════════════════════════════════════════╣
║  NEXT STEPS:                                                     ║
║                                                                  ║
║  1. Download model from Google Drive                            ║
║  2. Deploy with FastAPI server                                  ║
║  3. Configure Portkey webhook integration                        ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")
